In [1]:
"""Datenqualitäts-Checks für Favorita (Store Sales). Ausführen: uv run python src/eda_checks.py"""

'Datenqualitäts-Checks für Favorita (Store Sales). Ausführen: uv run python src/eda_checks.py'

In [2]:
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd().resolve().parent  # Skript oder Notebook
RAW = ROOT / "data" / "raw"
pd.set_option("display.width", 160)


def h(title):
    print(f"\n{'=' * 8} {title} {'=' * 8}")


train = pd.read_csv(RAW / "train.csv", parse_dates=["date"])
test = pd.read_csv(RAW / "test.csv", parse_dates=["date"])
stores = pd.read_csv(RAW / "stores.csv")
oil = pd.read_csv(RAW / "oil.csv", parse_dates=["date"])
hol = pd.read_csv(RAW / "holidays_events.csv", parse_dates=["date"])
trx = pd.read_csv(RAW / "transactions.csv", parse_dates=["date"])

## 1. Überblick

In [3]:
h("1. Überblick")
for name, df in [("train", train), ("test", test), ("stores", stores), ("oil", oil), ("holidays", hol), ("transactions", trx)]:
    print(f"{name:13s} {df.shape[0]:>9,} Zeilen  {df.shape[1]} Spalten")
print("train Zeitraum:", train.date.min().date(), "→", train.date.max().date())
print("test  Zeitraum:", test.date.min().date(), "→", test.date.max().date())
print("Filialen:", train.store_nbr.nunique(), " Produktfamilien:", train.family.nunique())


======== 1. Überblick ========
train         3,000,888 Zeilen  6 Spalten
test             28,512 Zeilen  5 Spalten
stores               54 Zeilen  5 Spalten
oil               1,218 Zeilen  2 Spalten
holidays            350 Zeilen  6 Spalten
transactions     83,488 Zeilen  3 Spalten
train Zeitraum: 2013-01-01 → 2017-08-15
test  Zeitraum: 2017-08-16 → 2017-08-31
Filialen: 54  Produktfamilien: 33


## 2. Fehlende Werte je Tabelle

In [4]:
h("2. Fehlende Werte je Tabelle")
for name, df in [("train", train), ("stores", stores), ("oil", oil), ("holidays", hol), ("transactions", trx)]:
    na = df.isna().sum()
    print(name, "→", dict(na[na > 0]) or "keine")


======== 2. Fehlende Werte je Tabelle ========
train → keine
stores → keine
oil → {'dcoilwtico': np.int64(43)}
holidays → keine
transactions → keine


## 3. Duplikate

In [5]:
h("3. Duplikate")
print("train: komplett identische Zeilen:", train.drop(columns="id").duplicated().sum())
print("train: doppelte Schlüssel (date, store, family):", train.duplicated(["date", "store_nbr", "family"]).sum())
print("transactions: doppelte (date, store):", trx.duplicated(["date", "store_nbr"]).sum())
print("oil: doppelte Daten:", oil.date.duplicated().sum())
print("holidays: doppelte (date, locale_name):", hol.duplicated(["date", "locale_name"]).sum(),
      "| mehrere Einträge am selben Tag:", hol.date.duplicated().sum())


======== 3. Duplikate ========


train: komplett identische Zeilen: 0


train: doppelte Schlüssel (date, store, family): 0
transactions: doppelte (date, store): 0
oil: doppelte Daten: 0
holidays: doppelte (date, locale_name): 7 | mehrere Einträge am selben Tag: 38


## 4. Lückenlose Zeitachse?

In [6]:
h("4. Lückenlose Zeitachse?")
full = pd.date_range(train.date.min(), train.date.max())
missing = full.difference(train.date.unique())
print("fehlende Kalendertage im train:", list(missing.strftime("%Y-%m-%d")))
per_day = train.groupby("date").size()
print("Zeilen pro Tag (erwartet 54*33=1782):", per_day.value_counts().to_dict())
print("oil: Kalendertage ohne Eintrag:", len(full.difference(oil.date)), "| Einträge mit NaN:", oil.dcoilwtico.isna().sum())


======== 4. Lückenlose Zeitachse? ========
fehlende Kalendertage im train: ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']
Zeilen pro Tag (erwartet 54*33=1782): {1782: 1684}
oil: Kalendertage ohne Eintrag: 482 | Einträge mit NaN: 43


## 5. Wertebereiche in sales / onpromotion

In [7]:
h("5. Wertebereiche in sales / onpromotion")
print(train[["sales", "onpromotion"]].describe().round(2).to_string())
print("negative sales:", (train.sales < 0).sum())
print("Anteil sales == 0:", round((train.sales == 0).mean(), 3))
print("Nicht ganzzahlige sales (Gewicht/kg-Artikel):", (train.sales % 1 != 0).sum())


======== 5. Wertebereiche in sales / onpromotion ========
            sales  onpromotion
count  3000888.00   3000888.00
mean       357.78         2.60
std       1102.00        12.22
min          0.00         0.00
25%          0.00         0.00
50%         11.00         0.00
75%        195.85         0.00
max     124717.00       741.00
negative sales: 0
Anteil sales == 0: 0.313


Nicht ganzzahlige sales (Gewicht/kg-Artikel): 462113


## 6. Filialen: Start der Verkäufe (Null-Phasen am Anfang)

In [8]:
h("6. Filialen: Start der Verkäufe (Null-Phasen am Anfang)")
daily = train.groupby(["store_nbr", "date"]).sales.sum().reset_index()
first_sale = daily[daily.sales > 0].groupby("store_nbr").date.min()
late = first_sale[first_sale > train.date.min() + pd.Timedelta(days=30)]
print(f"{len(late)} von 54 Filialen starten erst nach Tag 30 (neu eröffnet):")
print(late.dt.date.to_string())
zero_days = daily[daily.sales == 0].groupby("store_nbr").size()
print("Tage mit Gesamtumsatz 0 je Filiale (Top 8):", zero_days.sort_values(ascending=False).head(8).to_dict())


======== 6. Filialen: Start der Verkäufe (Null-Phasen am Anfang) ========


8 von 54 Filialen starten erst nach Tag 30 (neu eröffnet):
store_nbr
20    2015-02-13
21    2015-07-24
22    2015-10-09
29    2015-03-20
36    2013-05-09
42    2015-08-21
52    2017-04-20
53    2014-05-29
Tage mit Gesamtumsatz 0 je Filiale (Top 8): {52: 1566, 22: 1011, 42: 962, 21: 934, 29: 808, 20: 773, 53: 515, 36: 131}


## 7. Produktfamilien: Null-Reihen

In [9]:
h("7. Produktfamilien: Null-Reihen")
fam = train.groupby("family").sales.agg(total="sum", zero_share=lambda s: (s == 0).mean()).round(3)
print(fam.sort_values("zero_share", ascending=False).head(8).to_string())


======== 7. Produktfamilien: Null-Reihen ========


                               total  zero_share
family                                          
BOOKS                         6438.0       0.970
BABY CARE                    10051.0       0.941
SCHOOL AND OFFICE SUPPLIES  269316.0       0.741
HOME APPLIANCES              41601.0       0.735
LADIESWEAR                  651159.0       0.598
MAGAZINES                   266359.0       0.565
PET SUPPLIES                356584.0       0.546
HARDWARE                    103470.0       0.479


## 8. Promotion: wann ist onpromotion > 0?

In [10]:
h("8. Promotion: wann ist onpromotion > 0?")
promo_first = train[train.onpromotion > 0].date.min()
print("erster Tag mit onpromotion > 0:", promo_first.date())
print("Anteil Zeilen mit Promo vor/nach diesem Datum:",
      round((train[train.date >= promo_first].onpromotion > 0).mean(), 3))
print("Promo je Jahr (Anteil Zeilen > 0):")
print(train.assign(y=train.date.dt.year).groupby("y").onpromotion.apply(lambda s: round((s > 0).mean(), 3)).to_string())


======== 8. Promotion: wann ist onpromotion > 0? ========
erster Tag mit onpromotion > 0: 2014-04-01


Anteil Zeilen mit Promo vor/nach diesem Datum: 0.279
Promo je Jahr (Anteil Zeilen > 0):
y
2013    0.000
2014    0.100
2015    0.209
2016    0.351
2017    0.453


## 9. Feiertage

In [11]:
h("9. Feiertage")
print(hol.groupby(["type", "transferred"]).size().to_string())
print("Locale:", hol.locale.value_counts().to_dict())


======== 9. Feiertage ========
type        transferred
Additional  False           51
Bridge      False            5
Event       False           56
Holiday     False          209
            True            12
Transfer    False           12
Work Day    False            5
Locale: {'National': 174, 'Local': 152, 'Regional': 24}


## 10. Transactions vs. train

In [12]:
h("10. Transactions vs. train")
tr_pairs = set(zip(trx.date, trx.store_nbr))
tn_pairs = set(map(tuple, daily[daily.sales > 0][["date", "store_nbr"]].itertuples(index=False, name=None)))
print("Filial-Tage mit Umsatz, aber ohne Transaktionseintrag:", len(tn_pairs - tr_pairs))
print("Transaktionseinträge ohne Umsatz:", len(tr_pairs - tn_pairs))


======== 10. Transactions vs. train ========


Filial-Tage mit Umsatz, aber ohne Transaktionseintrag: 118
Transaktionseinträge ohne Umsatz: 0


## 11. Ausreißer: größte Tageswerte je (Filiale, Familie) vs. Median

In [13]:
h("11. Ausreißer: größte Tageswerte je (Filiale, Familie) vs. Median")
s = train[train.sales > 0].groupby(["store_nbr", "family"]).sales
ratio = (s.max() / s.median()).sort_values(ascending=False)
print(ratio.head(6).round(1).to_string())
top = train.nlargest(5, "sales")[["date", "store_nbr", "family", "sales"]]
print(top.to_string(index=False))


======== 11. Ausreißer: größte Tageswerte je (Filiale, Familie) vs. Median ========


store_nbr  family                    
39         MEATS                         653.6
18         CELEBRATION                   297.2
53         LINGERIE                      264.8
48         SCHOOL AND OFFICE SUPPLIES    202.8
47         SCHOOL AND OFFICE SUPPLIES    190.3
26         LIQUOR,WINE,BEER              187.6
      date  store_nbr    family      sales
2016-05-02          2 GROCERY I 124717.000
2016-10-07         39     MEATS  89576.360
2016-04-21         20 GROCERY I  87438.516
2016-04-18         45 GROCERY I  76090.000
2016-04-26          2 GROCERY I  63434.000


## 12. Erdbeben 16.04.2016 & Jahresende

In [14]:
h("12. Erdbeben 16.04.2016 & Jahresende")
tot = train.groupby("date").sales.sum()
print(tot["2016-04-10":"2016-04-24"].round(0).to_string())


======== 12. Erdbeben 16.04.2016 & Jahresende ========
date
2016-04-10    1008877.0
2016-04-11     665349.0
2016-04-12     651657.0
2016-04-13     672611.0
2016-04-14     546785.0
2016-04-15     667606.0
2016-04-16     862121.0
2016-04-17    1271834.0
2016-04-18    1345921.0
2016-04-19    1152089.0
2016-04-20    1062426.0
2016-04-21    1001080.0
2016-04-22     857059.0
2016-04-23    1022143.0
2016-04-24    1039370.0
